# DSA 504 — Class 13
## SQL Joins & Queries

**Date:** Monday, Oct 19
**Reading:** *Practical Business Analytics Using R and Python*, ch. 3

---

Class 12 worked entirely within one table (`sales`). Today we combine `sales` with `stores`, using the shared `store` column as the key that connects them — this is the entire point of splitting data across multiple tables in the first place.

**Heads up:** the `stores` table now has a fifth store, `Plattsburgh`, which has **no matching rows in `sales`** — added deliberately, so we have a real case to demonstrate what happens when a join doesn't find a match on one side.

### Learning goals
By the end of this class, you will be able to:
- Explain why data is often split across multiple tables instead of one large table
- Write an `INNER JOIN` to combine two tables on a shared key
- Write a `LEFT JOIN`, and explain how its results differ from an `INNER JOIN`
- Use table aliases to keep queries readable
- Combine a join with `GROUP BY` to aggregate across joined data


In [30]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("retail_sales.db")

print(pd.read_sql_query("SELECT * FROM stores", conn))
# print(pd.read_sql_query("SELECT * FROM sales", conn))
# tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
# print(tables)


         store            region  opened_year
0        Utica        Central NY         2014
1       Albany  Capital District         2011
2         Rome        Central NY         2018
3     Syracuse        Central NY         2016
4  Plattsburgh     North Country         2026


## 1. Why Split Data Across Tables At All?

We could have stuffed `region` and `opened_year` directly into every row of `sales` — but that means repeating "Central NY" thousands of times for every Utica transaction, and if a store's region ever changed, you'd have to update every single row instead of just one. Keeping store-level facts in their own small table, and transaction-level facts in another, avoids that repetition. A **join** is how you bring them back together only when you actually need both at once.


## 2. INNER JOIN — Matching Rows from Both Tables

An `INNER JOIN` returns only the rows where the join key finds a match in **both** tables.


In [33]:
query = """
SELECT sales.date, sales.store, sales.revenue, stores.region
FROM sales
INNER JOIN stores ON sales.store = stores.store
LIMIT 5
"""
result = pd.read_sql_query(query, conn)
result


,date,store,revenue,region
0,2025-04-22 00:00:00,Rome,402.50,Central NY
1,2025-04-12 00:00:00,Syracuse,3154.00,Central NY
2,2025-12-21 00:00:00,Utica,1299.06,Central NY
3,2025-02-11 00:00:00,Syracuse,5114.40,Central NY
4,2025-04-10 00:00:00,Syracuse,147.10,Central NY


**Reading the syntax:** `ON sales.store = stores.store` tells SQL which column connects the two tables — here, both tables happen to use the same column name (`store`), so we have to prefix each with its table name to avoid ambiguity.

### Table aliases — making this more readable

Writing `sales.` and `stores.` repeatedly gets tedious. Aliases let you give each table a short nickname for the rest of the query.


In [5]:
query = """
SELECT s.date, s.store, s.revenue, st.region
FROM sales AS s
INNER JOIN stores AS st ON s.store = st.store
LIMIT 5
"""
result = pd.read_sql_query(query, conn)
result


,date,store,revenue,region
0,2025-04-22 00:00:00,Rome,402.50,Central NY
1,2025-04-12 00:00:00,Syracuse,3154.00,Central NY
2,2025-12-21 00:00:00,Utica,1299.06,Central NY
3,2025-02-11 00:00:00,Syracuse,5114.40,Central NY
4,2025-04-10 00:00:00,Syracuse,147.10,Central NY


**Checking row counts — does an INNER JOIN lose anything here?**


In [6]:
print("Rows in sales:", pd.read_sql_query("SELECT COUNT(*) AS n FROM sales", conn)["n"][0])

query = """
SELECT COUNT(*) AS n
FROM sales AS s
INNER JOIN stores AS st ON s.store = st.store
"""
print("Rows after INNER JOIN:", pd.read_sql_query(query, conn)["n"][0])


Rows in sales: 4014
Rows after INNER JOIN: 4014


The counts match — every row in `sales` has a matching store in `stores`, so no rows were dropped by the join. That won't always be true, which is exactly why `LEFT JOIN` exists.


## 3. LEFT JOIN — Keeping Everything from the Left Table

A `LEFT JOIN` keeps **every row from the left (first) table**, even if there's no match on the right. Where there's no match, the right table's columns come back as `NULL` (which pandas shows as `NaN`).

Let's flip our query around and start from `stores` instead of `sales`, so `Plattsburgh` — which has no sales rows at all — becomes visible.


In [36]:
query = """
SELECT st.store, st.region, s.date, s.revenue
FROM stores AS st
LEFT JOIN sales AS s ON st.store = s.store
ORDER BY st.store
"""
result = pd.read_sql_query(query, conn)

print(result.head)

# Show just the Plattsburgh rows
result[result["store"] == "Plattsburgh"]


<bound method NDFrame.head of        store            region                 date  revenue
0     Albany  Capital District  2025-01-01 00:00:00   385.56
1     Albany  Capital District  2025-01-01 00:00:00  2750.94
2     Albany  Capital District  2025-01-02 00:00:00   370.76
3     Albany  Capital District  2025-01-02 00:00:00  3449.20
4     Albany  Capital District  2025-01-03 00:00:00   461.32
...      ...               ...                  ...      ...
4010   Utica        Central NY  2025-12-30 00:00:00  1176.35
4011   Utica        Central NY  2025-12-30 00:00:00  1292.48
4012   Utica        Central NY  2025-12-31 00:00:00   744.93
4013   Utica        Central NY  2025-12-31 00:00:00  1593.00
4014   Utica        Central NY  2025-12-31 00:00:00  3577.66

[4015 rows x 4 columns]>


,store,region,date,revenue
991,Plattsburgh,North Country,None,NaN


**Plattsburgh appears once, with `date` and `revenue` both `NULL`** — the row wasn't dropped (unlike what an `INNER JOIN` would do), but there was nothing to fill in from `sales`, so SQL fills the gap with `NULL`.


In [8]:
# Comparing row counts: INNER JOIN vs. LEFT JOIN starting from stores
inner_query = """
SELECT COUNT(*) AS n FROM stores AS st INNER JOIN sales AS s ON st.store = s.store
"""
left_query = """
SELECT COUNT(*) AS n FROM stores AS st LEFT JOIN sales AS s ON st.store = s.store
"""

print("INNER JOIN row count:", pd.read_sql_query(inner_query, conn)["n"][0])
print("LEFT JOIN row count:", pd.read_sql_query(left_query, conn)["n"][0])


INNER JOIN row count: 4014
LEFT JOIN row count: 4015


**Notice the LEFT JOIN count is exactly one higher** — that's the single `Plattsburgh` row with no sales match, which `INNER JOIN` silently drops entirely. This is the core practical difference: use `INNER JOIN` when you only want rows with data on both sides; use `LEFT JOIN` when you need to know about *every* row on the left, matched or not — for example, "which stores have never recorded a sale?" is a question only `LEFT JOIN` can answer.


In [9]:
# Using LEFT JOIN specifically to find stores with NO sales at all
query = """
SELECT st.store, st.region
FROM stores AS st
LEFT JOIN sales AS s ON st.store = s.store
WHERE s.store IS NULL
"""
result = pd.read_sql_query(query, conn)
result


,store,region
0,Plattsburgh,North Country


**`IS NULL`**, not `= NULL` — SQL requires this special syntax for checking nulls, the same way pandas needed `.isna()` instead of `== np.nan` back in Class 6. The same underlying idea shows up again here.


## 4. Combining a Join with GROUP BY

The real payoff of joins: now we can aggregate using a column that only exists in the *other* table — something no single-table query could do.


In [10]:
query = """
SELECT st.region, SUM(s.revenue) AS total_revenue
FROM sales AS s
INNER JOIN stores AS st ON s.store = st.store
GROUP BY st.region
ORDER BY total_revenue DESC
"""
result = pd.read_sql_query(query, conn)
result


,region,total_revenue
0,Central NY,4602739.67
1,Capital District,1477075.47


**Why `INNER JOIN` here, not `LEFT JOIN`?** Since `Plattsburgh` has no revenue to contribute to any region's total, including it with a `LEFT JOIN` wouldn't change the sums — but it's worth consciously choosing the join type based on what the query needs, not just defaulting to one.


In [11]:
# A more detailed example: average revenue per transaction, by region AND category
query = """
SELECT st.region, s.category, AVG(s.revenue) AS avg_revenue, COUNT(*) AS num_transactions
FROM sales AS s
INNER JOIN stores AS st ON s.store = st.store
GROUP BY st.region, s.category
ORDER BY st.region, avg_revenue DESC
"""
result = pd.read_sql_query(query, conn)
result


,region,category,avg_revenue,num_transactions
0,Capital District,electronics,3612.478649,185
1,Capital District,clothing,1540.565588,204
2,Capital District,home goods,1233.557902,205
3,Capital District,groceries,822.254975,201
4,Capital District,toys,389.484286,196
5,Central NY,electronics,3714.030103,581
6,Central NY,clothing,1543.471368,614
7,Central NY,home goods,1187.994338,604
8,Central NY,groceries,859.233052,629
9,Central NY,toys,402.000992,595


---
## Guided Practice

Write SQL queries against `sales` and `stores`, using `pd.read_sql_query()` to run them.


### Exercise 1 — Basic INNER JOIN
Write a query that returns `store`, `category`, `units_sold`, and `region` for the first 10 rows, joining `sales` and `stores` on `store`.


In [12]:
# Exercise 1 — your code here



### Exercise 2 — Filtering a joined query
Write a query that returns all columns, joined as above, but filtered to only rows where `region = 'Central NY'`. How many rows match?


In [13]:
# Exercise 2 — your code here



### Exercise 3 — LEFT JOIN and NULLs
Write a `LEFT JOIN` starting from `stores`, and print only the rows where there is no matching sale (hint: use `IS NULL` on a column that only exists in `sales`).


In [14]:
# Exercise 3 — your code here



### Exercise 4 — Join + GROUP BY
Write a query that computes total `units_sold` per `opened_year` (from `stores`), using a join between `sales` and `stores`.


In [15]:
# Exercise 4 — your code here



### Exercise 5 (stretch) — Three-way comparison
Write one query that shows, per region: total revenue, total units sold, and the number of distinct stores in that region (hint: `COUNT(DISTINCT st.store)`).


In [16]:
# Exercise 5 — your code here



---
## Solutions

Try each exercise yourself first. These are here for after class, or once you're stuck.


---
## Wrap-up

**Recap:** why data is split across multiple tables (avoiding repetition, easier updates); `INNER JOIN` for matched rows only; `LEFT JOIN` for keeping every row from the left table, with `NULL` filling in unmatched gaps; using `IS NULL` to find exactly those gaps; table aliases for readability; and combining a join with `GROUP BY` to aggregate using columns from either table.

**Common mistakes to watch for:**
- Forgetting the `ON` condition, which produces a cross join — every row of one table paired with every row of the other, usually a huge and meaningless result
- Using `WHERE s.store = NULL` instead of `WHERE s.store IS NULL`
- Defaulting to `INNER JOIN` without thinking about whether unmatched rows (like `Plattsburgh`) actually matter for the question being asked
- Ambiguous column errors when both tables share a column name and the query doesn't use a table prefix or alias

**Before next class (Oct 21):** Class 14 combines SQL with pandas directly — running a query and continuing the analysis in pandas — plus review for the practical midterm on Oct 26. **HW4 is assigned that day.**
